# 🇮🇳 Phase-IndianTriage — Pneumonia Screening with 4 Low-Cost Clinical Features

## Motivation
In Indian PHCs and CHCs, 4 basic parameters are ALWAYS available:

| Feature | Device | Cost (INR) |
|:---|:---|:---:|
| WBC Count | CBC Analyzer | ₹80–₹150 |
| SpO2 (O₂ Saturation) | Pulse Oximeter | Free (device ₹500) |
| Respiratory Rate | Manual count | ₹0 |
| Temperature (°F) | Digital thermometer | Free (device ₹100) |

## Design Principle
> **No imputation.** We train and evaluate ONLY on patients who have ALL 4 values recorded.
> This is the correct clinical approach — a doctor who measures these 4 parameters gets a prediction.
> A doctor who skips a measurement does NOT get a model output.

## Pipeline Overview
```
CELL 0  : Imports & Config
CELL 1  : Data Loading — Complete Cases Only (no imputation)
CELL 2  : Feature Distributions
CELL 3  : Stratified Train/Val/Test Split
CELL 4  : IMNCI Rule-Based Clinical Baseline
CELL 5  : ML Models (LR, Random Forest, Gradient Boosting, SVM, XGBoost)
CELL 6  : Model Comparison — AUC, Accuracy, Sensitivity, Specificity
CELL 7  : Best Model Analysis (Confusion Matrix, Feature Importance)
CELL 8  : Clinical Threshold Calibration (>=90% Sensitivity for India)
CELL 9  : 🏥 Live Multimodal Screening Interface (Doctor Demo)
CELL 10 : Final Summary Dashboard
```

In [ ]:
# --- CELL 0: Imports & Config ---
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — skipping.')

SEED = 42
np.random.seed(SEED)

BASE_DIR    = r'C:\2026\PneumoFusionNet\mimic\main'
DATASET_CSV = os.path.join(BASE_DIR, 'dataset', 'phase3_paired_scaleup_final.csv')
SAVE_DIR    = os.path.join(BASE_DIR, 'outputs', 'IndianTriage_4feature')
os.makedirs(SAVE_DIR, exist_ok=True)

FEATURES = ['wbc', 'spo2', 'respiratory_rate', 'temperature_f']
FEATURE_LABELS = {
    'wbc':              'WBC Count (×10³/µL)',
    'spo2':             'SpO2 (%)',
    'respiratory_rate': 'Respiratory Rate (br/min)',
    'temperature_f':    'Temperature (°F)'
}
TARGET = 'label'
print('✅ Setup complete.')

In [ ]:
# --- CELL 1: Data Loading — Complete Cases Only (NO Imputation) ---
df_raw = pd.read_csv(DATASET_CSV)
print(f'Raw dataset : {len(df_raw):,} samples')

print('\nMissing values per feature:')
for f in FEATURES:
    n_miss = df_raw[f].isnull().sum()
    print(f'  {FEATURE_LABELS[f]:<32}: {n_miss:>4} missing ({100*n_miss/len(df_raw):.1f}%)')

# ── Drop any row with ANY missing feature — no imputation ──
df = df_raw[FEATURES + [TARGET]].dropna().reset_index(drop=True)

print(f'\nComplete cases (all 4 features present): {len(df):,}')
print(f'  Dropped (at least 1 feature missing) : {len(df_raw) - len(df):,}')
print(f'\nLabel distribution (complete cases):')
vc = df[TARGET].value_counts()
print(f'  NORMAL    (0): {vc[0]:,}  ({100*vc[0]/len(df):.1f}%)')
print(f'  PNEUMONIA (1): {vc[1]:,}  ({100*vc[1]/len(df):.1f}%)')

print('\nDescriptive statistics (complete cases):')
display(df[FEATURES].describe().round(3))

In [ ]:
# --- CELL 2: Feature Distributions (Complete Cases) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(
    f'🇮🇳 4 Clinical Features — Complete Cases Only (N={len(df):,})\n'
    'No imputation: only patients with all 4 measurements recorded',
    fontsize=13, fontweight='bold'
)
palette = {0: '#2ecc71', 1: '#e74c3c'}
label_names = {0: 'Normal', 1: 'Pneumonia'}

for ax, feat in zip(axes.flatten(), FEATURES):
    for lbl, color in palette.items():
        vals = df[df[TARGET] == lbl][feat]
        ax.hist(vals, bins=35, alpha=0.65, color=color,
                label=label_names[lbl], edgecolor='white', linewidth=0.3)
    ax.set_xlabel(FEATURE_LABELS[feat], fontsize=11)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(FEATURE_LABELS[feat], fontsize=10)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved feature_distributions.png')

In [ ]:
# --- CELL 3: Stratified Train / Val / Test Split ---
X = df[FEATURES].values
y = df[TARGET].values

# 70 / 15 / 15 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f'Total  : {len(df):,}')
print(f'Train  : {len(X_train):,}  ({len(X_train)/len(df)*100:.0f}%)')
print(f'Val    : {len(X_val):,}   ({len(X_val)/len(df)*100:.0f}%)')
print(f'Test   : {len(X_test):,}   ({len(X_test)/len(df)*100:.0f}%)')
print(f'\nTest label balance: Normal={sum(y_test==0)} | Pneumonia={sum(y_test==1)}')
print('\n✅ No imputation applied. All splits contain only real measurements.')

In [ ]:
# --- CELL 4: IMNCI Rule-Based Clinical Baseline ---
# India NHM / WHO IMCI thresholds:
#   WBC  > 11.0  →  elevated (infection marker)
#   SpO2 < 95    →  low oxygen saturation
#   RR   > 22    →  tachypnea (fast breathing)
#   Temp > 100°F →  fever
# If ANY 1 flag fires → classify as PNEUMONIA

def imnci_predict(wbc, spo2, rr, temp):
    return int(
        wbc  > 11.0  or
        spo2 < 95.0  or
        rr   > 22.0  or
        temp > 100.0
    )

preds_rule = np.array([
    imnci_predict(*row) for row in df[FEATURES].values
])

rule_acc  = accuracy_score(y, preds_rule)
rule_auc  = roc_auc_score(y, preds_rule)
rule_f1   = f1_score(y, preds_rule)
cm_r      = confusion_matrix(y, preds_rule)
rule_sens = cm_r[1,1] / (cm_r[1,0] + cm_r[1,1])
rule_spec = cm_r[0,0] / (cm_r[0,0] + cm_r[0,1])

print('=' * 58)
print('  ⚖️  IMNCI Rule-Based Baseline (No ML, N=complete cases)')
print('=' * 58)
print(f'  Accuracy    : {rule_acc:.4f}  ({rule_acc*100:.1f}%)')
print(f'  AUC         : {rule_auc:.4f}')
print(f'  F1-Score    : {rule_f1:.4f}')
print(f'  Sensitivity : {rule_sens:.4f}  (Pneumonia recall — critical metric)')
print(f'  Specificity : {rule_spec:.4f}')
print()
print(classification_report(y, preds_rule, target_names=['Normal', 'Pneumonia']))

In [ ]:
# --- CELL 5: ML Model Training (No Imputation — Scaler Only) ---

def make_pipeline(clf):
    """Pipeline with StandardScaler only. No imputation."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    clf)
    ])

models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(random_state=SEED, max_iter=1000, C=1.0)
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(
            n_estimators=300, random_state=SEED,
            max_depth=6, min_samples_leaf=3,
            class_weight='balanced'
        )
    ),
    'Gradient Boosting': make_pipeline(
        GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05,
            max_depth=4, random_state=SEED
        )
    ),
    'SVM (RBF)': make_pipeline(
        CalibratedClassifierCV(
            SVC(kernel='rbf', C=1.0, random_state=SEED)
        )
    ),
}

if XGBOOST_AVAILABLE:
    models['XGBoost'] = make_pipeline(
        XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            use_label_encoder=False, eval_metric='logloss',
            random_state=SEED, verbosity=0
        )
    )

results = {}
print('Training on complete cases only (N_train={})...'.format(len(X_train)))
print('-' * 62)

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_test)[:, 1]
    preds = pipe.predict(X_test)

    auc  = roc_auc_score(y_test, probs)
    acc  = accuracy_score(y_test, preds)
    f1   = f1_score(y_test, preds)
    cm   = confusion_matrix(y_test, preds)
    sens = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0]+cm[1,1]) > 0 else 0
    spec = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0]+cm[0,1]) > 0 else 0

    results[name] = dict(auc=auc, acc=acc, f1=f1,
                         sens=sens, spec=spec,
                         probs=probs, preds=preds, pipe=pipe)
    print(f'  {name:<22}  AUC={auc:.4f}  Acc={acc:.4f}  Sens={sens:.4f}  Spec={spec:.4f}')

print('-' * 62)
print('✅ Training complete. No missing value imputation was used.')

In [ ]:
# --- CELL 6: Model Comparison Table & ROC Curves ---

rows = []
for name, r in results.items():
    rows.append(dict(Model=name, AUC=r['auc'], Accuracy=r['acc'],
                     F1=r['f1'], Sensitivity=r['sens'], Specificity=r['spec']))
rows.append(dict(Model='⚖️ IMNCI Rules', AUC=rule_auc, Accuracy=rule_acc,
                 F1=rule_f1, Sensitivity=rule_sens, Specificity=rule_spec))

comp_df = pd.DataFrame(rows).sort_values('AUC', ascending=False).reset_index(drop=True)
print('📊 Model Comparison (Test Set — Complete Cases Only):')
display(comp_df.style.format({
    'AUC': '{:.4f}', 'Accuracy': '{:.4f}',
    'F1': '{:.4f}', 'Sensitivity': '{:.4f}', 'Specificity': '{:.4f}'
}).bar(subset=['AUC'], color='#3498db').highlight_max(subset=['AUC','Sensitivity'], color='#d4efdf'))

comp_df.to_csv(os.path.join(SAVE_DIR, 'model_comparison.csv'), index=False)

# ROC Curve
colors = ['#e74c3c','#3498db','#9b59b6','#f39c12','#1abc9c']
fig, ax = plt.subplots(figsize=(9, 7))

for (name, r), col in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['probs'])
    ax.plot(fpr, tpr, lw=2.2, color=col, label=f"{name} (AUC={r['auc']:.4f})")

ax.scatter(1-rule_spec, rule_sens, color='black', s=120, zorder=5,
           marker='D', label=f'IMNCI Rules (AUC={rule_auc:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('🇮🇳 ROC Curves — 4-Feature Indian Triage\n(No imputation, complete cases only)', fontsize=12)
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'roc_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved roc_comparison.png')

In [ ]:
# --- CELL 7: Best Model — Confusion Matrix & Feature Importance ---
best_name = max(results, key=lambda k: results[k]['auc'])
best      = results[best_name]
print(f'🏆 Best Model : {best_name}  |  AUC = {best["auc"]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Best Model: {best_name}  (AUC={best["auc"]:.4f})', fontsize=12, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, best['preds'])
ConfusionMatrixDisplay(cm, display_labels=['Normal','Pneumonia']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion Matrix (Test Set)', fontsize=11)

# Feature Importance / Coefficients
clf = best['pipe'].named_steps['clf']
feat_labels = [FEATURE_LABELS[f] for f in FEATURES]

if hasattr(clf, 'feature_importances_'):
    imp = clf.feature_importances_
    idx = np.argsort(imp)
    colors_bar = ['#e74c3c' if imp[i]==imp.max() else '#3498db' for i in idx]
    axes[1].barh([feat_labels[i] for i in idx], imp[idx], color=colors_bar)
    axes[1].set_xlabel('Feature Importance (Gini)', fontsize=11)
    axes[1].set_title('Feature Importance', fontsize=11)
elif hasattr(clf, 'coef_'):
    coefs = np.abs(clf.coef_[0])
    idx   = np.argsort(coefs)
    axes[1].barh([feat_labels[i] for i in idx], coefs[idx], color='#3498db')
    axes[1].set_xlabel('|Coefficient|', fontsize=11)
    axes[1].set_title('Feature Coefficients (Absolute)', fontsize=11)
else:
    axes[1].text(0.5, 0.5, 'Feature importance not available',
                 ha='center', va='center', transform=axes[1].transAxes)

axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'best_model_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, best['preds'], target_names=['Normal','Pneumonia']))

In [ ]:
# --- CELL 8: Clinical Threshold Calibration ---
# India clinical goal: Sensitivity >= 90%
# Better to over-refer (false positive) than miss pneumonia (false negative)

probs_best = best['probs']
fpr, tpr, thresholds = roc_curve(y_test, probs_best)

# Youden's J (balanced)
youden_idx    = np.argmax(tpr - fpr)
thresh_youden = float(thresholds[youden_idx])

# Clinical (India NHM): first threshold achieving >= 90% sensitivity
clinical_idxs = np.where(tpr >= 0.90)[0]
thresh_clin   = float(thresholds[clinical_idxs[0]]) if len(clinical_idxs) > 0 else thresh_youden

def eval_thresh(thresh, label):
    p  = (probs_best >= thresh).astype(int)
    cm = confusion_matrix(y_test, p)
    ac = accuracy_score(y_test, p)
    se = cm[1,1]/(cm[1,0]+cm[1,1])
    sp = cm[0,0]/(cm[0,0]+cm[0,1])
    f1 = f1_score(y_test, p)
    print(f'  {label:<38} thresh={thresh:.3f} | Acc={ac:.3f} Sens={se:.3f} Spec={sp:.3f} F1={f1:.3f}')
    return thresh, ac, se, sp

print('=' * 72)
print(f'  Threshold Calibration — {best_name}')
print('=' * 72)
eval_thresh(0.500,         'Default (0.500)                   ')
eval_thresh(thresh_youden, 'Youden-J (balanced accuracy)      ')
t, _, cs, _ = eval_thresh(thresh_clin,  'Clinical India (>=90% Sensitivity)')

CLINICAL_THRESHOLD = thresh_clin
print(f'\n  ✅ Deployment threshold = {CLINICAL_THRESHOLD:.3f}')
print(f'     Achieved sensitivity  = {cs:.1%} on test set')

In [ ]:
# --- CELL 9: 🏥 Live Multimodal Screening Interface ---
# A doctor/technician enters the 4 measured values and gets
# an AI risk prediction + IMNCI-aligned recommendation.

BEST_PIPE  = best['pipe']

def screen_patient(
    wbc: float,
    spo2: float,
    respiratory_rate: float,
    temperature_f: float,
    patient_id: str = 'N/A',
    age: int = None,
    gender: str = None,
    xray_path: str = None
):
    """
    🇮🇳 Indian PHC Pneumonia Triage Predictor

    All 4 parameters are REQUIRED (no imputation).
    If a measurement is unavailable, the test cannot be run.
    """
    # Validate — no NaN allowed
    for name, val in [('WBC', wbc), ('SpO2', spo2),
                      ('Respiratory Rate', respiratory_rate),
                      ('Temperature', temperature_f)]:
        if val is None or (isinstance(val, float) and np.isnan(val)):
            print(f'❌ ERROR: {name} is missing. All 4 measurements are required.')
            return None

    X_input = np.array([[wbc, spo2, respiratory_rate, temperature_f]])
    prob    = BEST_PIPE.predict_proba(X_input)[0, 1]

    # ── Clinical flag checks ──
    flags = []
    if wbc  > 15.0:  flags.append('⚠️ Very High WBC (>15)')
    elif wbc > 11.0: flags.append('🟡 Elevated WBC (>11)')
    if spo2  < 90:   flags.append('🔴 CRITICAL SpO2 (<90%)')
    elif spo2 < 95:  flags.append('🟡 Low SpO2 (<95%)')
    if respiratory_rate > 30:  flags.append('🔴 Very High RR (>30)')
    elif respiratory_rate > 22: flags.append('🟡 Elevated RR (>22)')
    if temperature_f > 101.0:  flags.append('🔴 High Fever (>101°F)')
    elif temperature_f > 100.0: flags.append('🟡 Low-Grade Fever (>100°F)')

    severe = [f for f in flags if '🔴' in f]

    # ── Risk tier ──
    if len(severe) >= 1 or prob >= 0.80:
        tier    = '🔴 SEVERE — REFER IMMEDIATELY'
        action  = 'Urgent referral to District Hospital / Emergency'
        rx      = 'IV Amoxicillin + Gentamicin (pre-referral dose per NHM protocol)'
    elif prob >= CLINICAL_THRESHOLD:
        tier    = '🟡 PNEUMONIA — TREAT AT PHC'
        action  = 'Treat at PHC. Oral antibiotics. Review in 48 hours.'
        rx      = 'Oral Amoxicillin 40–45 mg/kg/day × 5 days (WHO/NHM)'
    else:
        tier    = '🟢 NORMAL — NO PNEUMONIA'
        action  = 'No antibiotic. Supportive care (ORS, paracetamol if fever). Return if worsens.'
        rx      = 'None required'

    # ── Print report ──
    print('━' * 62)
    print('  🏥 INDIAN PHC — PNEUMONIA TRIAGE REPORT')
    print('━' * 62)
    print(f'  Patient ID         : {patient_id}')
    if age:    print(f'  Age                : {age} years')
    if gender: print(f'  Gender             : {gender}')
    if xray_path: print(f'  Chest X-Ray        : {os.path.basename(xray_path)}')
    print('─' * 62)
    print('  INPUT MEASUREMENTS (real values, no imputation):')
    print(f'    WBC Count          : {wbc:.1f} ×10³/µL')
    print(f'    SpO2               : {spo2:.0f} %')
    print(f'    Respiratory Rate   : {respiratory_rate:.0f} breaths/min')
    print(f'    Temperature        : {temperature_f:.1f} °F')
    print('─' * 62)
    print(f'  AI MODEL            : {best_name}')
    print(f'  Pneumonia Prob      : {prob:.1%}')
    print(f'  Clinical Threshold  : {CLINICAL_THRESHOLD:.3f} (India >=90% Sens)')
    print(f'  Clinical Flags      : {flags if flags else ["None"]}')
    print('─' * 62)
    print(f'  RISK TIER  →  {tier}')
    print(f'  ACTION     →  {action}')
    print(f'  ANTIBIOTIC →  {rx}')
    print('━' * 62)

    return dict(probability=round(prob, 4), tier=tier, flags=flags,
                action=action, antibiotic=rx)

# ────────────────────────────────────────────────────
print('\n📋 TEST CASE 1 — Classic Bacterial Pneumonia')
screen_patient(wbc=14.5, spo2=91, respiratory_rate=26, temperature_f=101.2,
               patient_id='OPD-2024-001', age=45, gender='M')

print('\n📋 TEST CASE 2 — Healthy Patient / Common Cold')
screen_patient(wbc=7.2, spo2=98, respiratory_rate=17, temperature_f=98.4,
               patient_id='OPD-2024-002', age=28, gender='F')

print('\n📋 TEST CASE 3 — Borderline Case (Mild Pneumonia?)')
screen_patient(wbc=11.8, spo2=94, respiratory_rate=23, temperature_f=100.2,
               patient_id='OPD-2024-003', age=38, gender='M')

print('\n📋 TEST CASE 4 — Severe Emergency Referral')
screen_patient(wbc=18.5, spo2=84, respiratory_rate=34, temperature_f=102.5,
               patient_id='EMRG-2024-004', age=72, gender='F')

print('\n📋 TEST CASE 5 — Missing Value Rejection (no imputation)')
screen_patient(wbc=10.2, spo2=None, respiratory_rate=20, temperature_f=99.1,
               patient_id='PHC-2024-005')

In [ ]:
# --- CELL 10: Final Summary Dashboard ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    '🇮🇳 Indian-Context Pneumonia Triage — Final Summary\n'
    '4 Features: WBC + SpO2 + Respiratory Rate + Temperature  |  No Imputation',
    fontsize=13, fontweight='bold'
)

# 1. AUC Bar
ax = axes[0, 0]
names = list(results.keys()) + ['IMNCI Rules']
aucs  = [results[m]['auc'] for m in results] + [rule_auc]
bar_cols = ['#e74c3c' if n == best_name else '#3498db' for n in results] + ['#7f8c8d']
bars = ax.barh(names, aucs, color=bar_cols)
ax.axvline(0.80, color='orange', linestyle='--', lw=1.2, label='AUC=0.80 target')
for bar, v in zip(bars, aucs):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)
ax.set_xlabel('ROC-AUC', fontsize=10)
ax.set_title('Model AUC Comparison', fontsize=10)
ax.set_xlim(0.40, 1.05)
ax.legend(fontsize=8)
ax.spines[['top','right']].set_visible(False)

# 2. Sensitivity vs Specificity scatter
ax = axes[0, 1]
for i, (name, r) in enumerate(results.items()):
    c = '#e74c3c' if name == best_name else '#3498db'
    ax.scatter(r['spec'], r['sens'], s=110, color=c, zorder=3)
    ax.annotate(name.split()[0], (r['spec'], r['sens']),
                xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.scatter(rule_spec, rule_sens, marker='D', s=120, color='gray', zorder=3, label='IMNCI')
ax.axhline(0.90, color='orange', linestyle='--', lw=1.2, label='90% Sens target')
ax.set_xlabel('Specificity', fontsize=10)
ax.set_ylabel('Sensitivity', fontsize=10)
ax.set_title('Sensitivity vs Specificity', fontsize=10)
ax.legend(fontsize=8)
ax.spines[['top','right']].set_visible(False)

# 3. Feature tier availability table
ax = axes[1, 0]
tiers  = ['ASHA (Village)', 'PHC (Block)', 'CHC (Taluk)', 'District Hospital']
avail  = [[0,0,0,0],[1,0,0,0],[1,1,1,1],[1,1,1,1]]
feats  = ['WBC','SpO2','RR','Temp']
table  = ax.table(
    cellText=[['✅' if v else '❌' for v in row] for row in avail],
    rowLabels=tiers, colLabels=feats,
    cellLoc='center', loc='center'
)
for (r, c), cell in table.get_celld().items():
    if r > 0 and c >= 0:
        cell.set_facecolor('#d4efdf' if avail[r-1][c] else '#fadbd8')
    cell.set_fontsize(10)
ax.axis('off')
ax.set_title('Feature Availability by India Healthcare Tier', fontsize=10)

# 4. Best model confusion matrix
ax = axes[1, 1]
cm = confusion_matrix(y_test, best['preds'])
ConfusionMatrixDisplay(cm, display_labels=['Normal','Pneumonia']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Best: {best_name}\nAUC={best["auc"]:.4f}', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'indian_triage_summary_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*65)
print('  🇮🇳 FINAL RESULTS — Indian PHC Triage (No Imputation)')
print('='*65)
print(f'  Dataset       : {len(df):,} complete cases (all 4 features real)')
print(f'  Best Model    : {best_name}')
print(f'  Test AUC      : {best["auc"]:.4f}')
print(f'  Accuracy      : {best["acc"]*100:.1f}%')
print(f'  Sensitivity   : {best["sens"]*100:.1f}%  (Pneumonia Recall)')
print(f'  Specificity   : {best["spec"]*100:.1f}%  (Normal Recall)')
print(f'  IMNCI Rules   : AUC={rule_auc:.4f}')
print(f'  Clinical Thresh: {CLINICAL_THRESHOLD:.3f} (India >=90% Sens)')
print('='*65)
print('  ✅ Next Step: Combine with Chest X-Ray Grad-CAM')
print('     for PneumoFusionNet-India multimodal deployment.')
print('='*65)